# VHAGAR T2 head-to-head (Colab)

Prithvi vs U-Net vs RBR-threshold on your **real held-out fires**, scored per fire as
skill over predict-all-burned, with **paired bootstrap confidence intervals** on the
differences. A margin counts only if its CI excludes zero.

**Before you run**: set *Runtime -> Change runtime type -> GPU*, and put your MTBS
6-band sample cache (the `.npz` files) somewhere in Google Drive.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Get VHAGAR
Public repo clones directly. Private repo: paste a GitHub token when prompted, or
upload the repo to Drive and point `REPO_DIR` at it instead of cloning.


In [ ]:
REPO = 'https://github.com/Ibekwemmanuel7/VHAGAR.git'   # <- your repo
REPO_DIR = '/content/VHAGAR'
import os
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO $REPO_DIR || echo 'clone failed (private? use a token)'
# Colab already ships torch + numpy + scipy + sklearn + pandas + pyarrow.
# Do NOT pip-install numpy here: upgrading it breaks the running kernel's ABI.
import sys; sys.path.insert(0, f'{REPO_DIR}/src')
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())


## 3. Config
Point `CACHE_DIR` at your `.npz` cache. Leave `PRITHVI_DIR = None` to run U-Net vs RBR
only; set it to a folder of `<event_id>.npy` burned masks to add the Prithvi leg.


In [ ]:
CACHE_DIR   = '/content/drive/MyDrive/vhagar/t2_cache'   # <- your .npz cache
PATTERN     = 'mtbs_*_w15bg.npz'
PRITHVI_DIR = None   # e.g. '/content/drive/MyDrive/vhagar/prithvi_masks'
VAL_FRAC, TEST_FRAC = 0.15, 0.15
EPOCHS, N_BOOT, SEED = 30, 10000, 0


## 4. Load the fires


In [ ]:
import glob, numpy as np
from vhagar.datasets.burned_area import T2Sample
paths = sorted(glob.glob(f'{CACHE_DIR}/{PATTERN}'))
samples = {}
for p in paths:
    s = T2Sample.load(p)
    if s.is_usable:
        samples[s.event_id] = s
print(f'{len(samples)} usable fires from {len(paths)} files')
assert len(samples) >= 3, 'need at least 3 usable fires for a grouped split'


## 5. Optional: load Prithvi masks
Produced by the out-of-repo TerraTorch inference + `t2_prithvi.stitch_chip_predictions`,
one `<event_id>.npy` boolean burned mask per fire on the sample's grid.


In [ ]:
prithvi = None
if PRITHVI_DIR:
    import os
    prithvi = {}
    for eid in samples:
        f = os.path.join(PRITHVI_DIR, f'{eid}.npy')
        if os.path.exists(f):
            prithvi[eid] = np.load(f)
    print(f'loaded {len(prithvi)} Prithvi masks')


## 6. Run the head-to-head
Trains the U-Net on the training fires (GPU), scores every model on the identical
held-out fires through the same metric, and bootstraps the paired differences.


In [ ]:
from vhagar.eval.t2_headtohead import head_to_head
rep = head_to_head(samples, prithvi_pred_by_event=prithvi,
                   val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED,
                   n_boot=N_BOOT, unet_kw={'epochs': EPOCHS})

print(f"held-out fires: {rep['n_test_fires']}  ({', '.join(rep['test_fires'])})\n")
print('mean per-fire skill (F1 over predict-all-burned):')
for m, v in rep['mean_skill'].items():
    print(f'  {m:8s} {v:+.3f}')
print('\npaired bootstrap differences (95% CI):')
for d in rep['paired_diffs']:
    tag = 'SEPARABLE' if d.separable else 'not separable'
    print(f'  {d.a:8s} - {d.b:8s}  {d.mean_diff:+.3f}  '
          f'[{d.ci_lo:+.3f}, {d.ci_hi:+.3f}]  P({d.a}>{d.b})={d.prob_a_better:.2f}  {tag}')
for n in rep['notes']:
    print(' note:', n)


## How to read it
- **mean skill**: higher is better; the RBR threshold is the baseline every model must beat.
- **SEPARABLE** means the 95% CI on the per-fire difference excludes zero, so the margin is
  real, not fold-variance. On ~20 fires the honest verdict is often *not separable*; that is
  information, not failure. Scaling to ~60 fires is what sharpens it.

## Producing the Prithvi masks (separate GPU step)
In the repo, export chips and run TerraTorch, then stitch back to per-fire masks:
```
python -m vhagar.cli t2-prithvi-export --cache-dir <cache> --out-dir chips
# fine-tune + infer Prithvi-EO on `chips` with TerraTorch (GPU)
# then: from vhagar.eval.t2_prithvi import stitch_chip_predictions
#       save each event's mask to PRITHVI_DIR/<event_id>.npy and re-run cell 6
```
